# Early Fusion with CodeT5+ Encoder Only
## Efficient Multimodal Classification

In [24]:
import os
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, T5EncoderModel, AutoImageProcessor, ViTModel
from PIL import Image
from tqdm import tqdm
import numpy as np
import pandas as pd

In [25]:
# Configuration
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR = "../checkpoints/early_fusion_ct5p_encoder/"
BATCH_SIZE = 8
EPOCHS = 3
LR = 2e-5

CT5P_CKPT = "../checkpoints/ct5p_only/checkpoint-2322/"
VIT_CKPT = "../checkpoints/vit_only/deit_epoch_3.pt"

TEXT_DIR = "../Text_Files/Train"
IMAGE_DIR = "../snapshots/Train"
TEST_BASE = "../snapshots"

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [26]:
class CodeT5ForSequenceClassification(nn.Module):
    """
    Custom CodeT5+ model for sequence classification using only the encoder
    """
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.encoder = T5EncoderModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.encoder.config.d_model, num_labels)
        self.num_labels = num_labels
        
    def forward(self, input_ids, attention_mask=None, labels=None):
        # Get encoder outputs
        outputs = self.encoder(
            input_ids=input_ids, 
            attention_mask=attention_mask,
            return_dict=True
        )
        
        # Use mean pooling over sequence dimension
        sequence_output = outputs.last_hidden_state
        
        # Apply attention mask for mean pooling
        if attention_mask is not None:
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(sequence_output.size()).float()
            sum_embeddings = torch.sum(sequence_output * input_mask_expanded, 1)
            sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
            pooled_output = sum_embeddings / sum_mask
        else:
            pooled_output = sequence_output.mean(dim=1)
        
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))
        
        return {'loss': loss, 'logits': logits}

In [27]:
# Load CodeT5+ Encoder Only
print("Loading CodeT5+ Encoder...")
tokenizer = AutoTokenizer.from_pretrained("Salesforce/codet5p-220m")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

from transformers import T5EncoderModel

# 1️⃣ Load your fine-tuned classification model
model = CodeT5ForSequenceClassification("Salesforce/codet5p-220m", 2)
checkpoint = torch.load(os.path.join(CT5P_CKPT, "pytorch_model.bin"), map_location=DEVICE)
model.load_state_dict(checkpoint, strict=False)
model.to(DEVICE)
model.eval()

print("✓ Full fine-tuned CodeT5+ classification model loaded.")

# 2️⃣ Extract the encoder weights only
encoder_state_dict = model.encoder.state_dict()

# 3️⃣ Load them into a fresh encoder model
text_model = T5EncoderModel.from_pretrained("Salesforce/codet5p-220m")
text_model.load_state_dict(encoder_state_dict, strict=False)
text_model.to(DEVICE)
text_model.eval()

print("✓ Encoder extracted successfully from fine-tuned model.")

Loading CodeT5+ Encoder...
✓ Full fine-tuned CodeT5+ classification model loaded.
✓ Full fine-tuned CodeT5+ classification model loaded.
✓ Encoder extracted successfully from fine-tuned model.
✓ Encoder extracted successfully from fine-tuned model.


In [28]:
# Load ViT model
print("Loading ViT model...")
image_processor = AutoImageProcessor.from_pretrained("facebook/deit-base-patch16-224")

# Load your trained classification model first
from transformers import ViTForImageClassification
trained_model = ViTForImageClassification.from_pretrained(
    "facebook/deit-base-patch16-224",
    num_labels=2,
    ignore_mismatched_sizes=True
)
trained_model.load_state_dict(torch.load(VIT_CKPT, map_location=DEVICE))
trained_model.to(DEVICE)

# Extract just the ViT base model for embeddings
vit_model = trained_model.vit
vit_model.to(DEVICE)
vit_model.eval()
print("✓ ViT model loaded successfully")

Loading ViT model...


Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.
Some weights of ViTForImageClassification were not initialized from the model checkpoint at facebook/deit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([2, 768]) in the model instantiated
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([2]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of ViTForImageClassification were not initialized from the model checkpoint at facebook/deit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([1000, 7

✓ ViT model loaded successfully


In [29]:
class FusionDataset(Dataset):
    def __init__(self, text_dir, image_dir, tokenizer, image_processor):
        self.text_paths = []
        self.image_paths = []
        self.labels = []
        self.tokenizer = tokenizer
        self.image_processor = image_processor

        # Scan text folders
        for label_folder in sorted(os.listdir(text_dir)):
            label_path = os.path.join(text_dir, label_folder)
            if not os.path.isdir(label_path):
                continue
            label = int(label_folder.split("_")[1])  # e.g., "Label_0" -> 0
            for txt_file in sorted(os.listdir(label_path)):
                if txt_file.endswith(".txt"):
                    self.text_paths.append(os.path.join(label_path, txt_file))
                    self.labels.append(label)

        # Scan image folders
        self.image_paths = []
        for label_folder in sorted(os.listdir(image_dir)):
            label_path = os.path.join(image_dir, label_folder)
            if not os.path.isdir(label_path):
                continue
            for img_file in sorted(os.listdir(label_path)):
                if img_file.lower().endswith((".png", ".jpg", ".jpeg")):
                    self.image_paths.append(os.path.join(label_path, img_file))

        # Ensure text_paths and image_paths are aligned
        assert len(self.text_paths) == len(self.image_paths), "Text and image counts must match!"
        print(f"✓ Loaded {len(self.text_paths)} samples")

    def __len__(self):
        return len(self.text_paths)

    def __getitem__(self, idx):
        # ----- TEXT -----
        with open(self.text_paths[idx], "r", encoding='utf-8', errors='ignore') as f:
            text = f.read()
        encoding = self.tokenizer(
            text, 
            return_tensors="pt", 
            truncation=True, 
            padding="max_length", 
            max_length=512
        )
        input_ids = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)

        # ----- IMAGE -----
        image = Image.open(self.image_paths[idx]).convert("RGB")
        image_tensor = self.image_processor(images=image, return_tensors="pt")
        for k in image_tensor:
            image_tensor[k] = image_tensor[k].squeeze(0)

        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return input_ids, attention_mask, image_tensor, label

In [30]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

In [31]:
class FusionClassifier(nn.Module):
    def __init__(self, text_model, vit_model, hidden_dim=512, num_classes=2):
        super().__init__()
        self.text_model = text_model
        self.vit_model = vit_model

        # Freeze backbone models
        for p in self.text_model.parameters():
            p.requires_grad = False
        for p in self.vit_model.parameters():
            p.requires_grad = False

        # Get embedding dimensions
        text_emb_dim = text_model.config.d_model
        vit_emb_dim = vit_model.config.hidden_size

        # Individual classifiers for late fusion
        self.text_head = nn.Linear(text_emb_dim, num_classes)
        self.image_head = nn.Linear(vit_emb_dim, num_classes)

        # Fused classifier
        self.classifier = nn.Sequential(
            nn.Linear(text_emb_dim + vit_emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, input_ids, attention_mask, image_tensor, return_all=False):
        # ----- TEXT EMBEDDING -----
        text_outputs = self.text_model(input_ids=input_ids, attention_mask=attention_mask)
        text_embeddings = text_outputs.last_hidden_state  # [B, seq, hid]

        if attention_mask is not None:
            mask_expanded = attention_mask.unsqueeze(-1).expand(text_embeddings.size()).float()
            sum_embeddings = torch.sum(text_embeddings * mask_expanded, dim=1)
            sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)
            text_cls = sum_embeddings / sum_mask
        else:
            text_cls = text_embeddings.mean(dim=1)

        text_logits = self.text_head(text_cls)

        # ----- IMAGE EMBEDDING -----
        image_outputs = self.vit_model(**image_tensor)
        image_cls = image_outputs.last_hidden_state[:, 0, :]
        image_logits = self.image_head(image_cls)

        # ----- FUSED LOGITS -----
        fused = torch.cat([text_cls, image_cls], dim=1)
        fused_logits = self.classifier(fused)

        # Return everything for late fusion
        if return_all:
            return fused_logits, text_logits, image_logits

        return fused_logits


In [32]:
# Load dataset
print("Loading training dataset...")
dataset = FusionDataset(TEXT_DIR, IMAGE_DIR, tokenizer, image_processor)
train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# Initialize model
model = FusionClassifier(text_model, vit_model).to(DEVICE)
optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

print(f"✓ Model initialized on {DEVICE}")
print(f"✓ Training samples: {len(dataset)}")
print(f"✓ Batch size: {BATCH_SIZE}")
print(f"✓ Total parameters: {sum(p.numel() for p in model.classifier.parameters()):,}")

Loading training dataset...
✓ Loaded 6190 samples
✓ Model initialized on cuda
✓ Training samples: 6190
✓ Batch size: 8
✓ Total parameters: 787,970


In [10]:
print("Starting training...")

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    
    for batch in progress_bar:
        input_ids, attention_mask, image_tensor, labels = batch
        
        # Move to device
        input_ids = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)
        labels = labels.to(DEVICE)
        for k in image_tensor:
            image_tensor[k] = image_tensor[k].to(DEVICE)

        # Forward pass
        logits = model(input_ids, attention_mask, image_tensor)
        loss = criterion(logits, labels)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})
    
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} | Avg Loss: {avg_loss:.4f}")

    # Save checkpoint
    ckpt_path = os.path.join(OUTPUT_DIR, f"fusion_ct5p_norm_encoder_epoch_{epoch+1}.pt")
    torch.save(model.state_dict(), ckpt_path)
    print(f"✓ Checkpoint saved: {ckpt_path}")

print("✓ Training completed!")

Starting training...


Epoch 1/3: 100%|██████████| 774/774 [01:30<00:00,  8.54it/s, loss=0.0188]


Epoch 1/3 | Avg Loss: 0.0975
✓ Checkpoint saved: ../checkpoints/early_fusion_ct5p_encoder/fusion_ct5p_norm_encoder_epoch_1.pt


Epoch 2/3: 100%|██████████| 774/774 [01:30<00:00,  8.54it/s, loss=0.0129]


Epoch 2/3 | Avg Loss: 0.0663
✓ Checkpoint saved: ../checkpoints/early_fusion_ct5p_encoder/fusion_ct5p_norm_encoder_epoch_2.pt


Epoch 3/3: 100%|██████████| 774/774 [01:30<00:00,  8.54it/s, loss=0.0426]


Epoch 3/3 | Avg Loss: 0.0630
✓ Checkpoint saved: ../checkpoints/early_fusion_ct5p_encoder/fusion_ct5p_norm_encoder_epoch_3.pt
✓ Training completed!


# Evaluation

In [23]:
def test_on_dataset(text_dir, image_dir, tokenizer, image_processor, fusion_model, batch_size=4):
    """Test the fusion model on a single dataset with late fusion saving.

    For each text-image pair this computes class probabilities for the text and
    image heads, picks the prediction whose max-probability (confidence) is
    higher, and uses that as the combined late-fusion prediction.
    """

    dataset = FusionDataset(text_dir, image_dir, tokenizer, image_processor)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    fusion_model.eval()

    total_correct = 0
    total_samples = 0

    # storage for saving results
    all_labels = []
    all_text_preds = []
    all_image_preds = []
    all_combined_preds = []
    all_text_conf = []
    all_image_conf = []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Testing"):
            input_ids, attention_mask, image_tensor, labels = batch

            # Move to device
            input_ids = input_ids.to(DEVICE)
            attention_mask = attention_mask.to(DEVICE)
            labels = labels.to(DEVICE)
            for k in image_tensor:
                image_tensor[k] = image_tensor[k].to(DEVICE)

            # Forward pass
            logits_fusion, text_logits, image_logits = fusion_model(
                input_ids, attention_mask, image_tensor, return_all=True
            )

            # normal fusion prediction (unused for late fusion accuracy)
            fusion_preds = torch.argmax(logits_fusion, dim=1)

            # late fusion components
            text_probs = torch.softmax(text_logits, dim=1)
            image_probs = torch.softmax(image_logits, dim=1)

            text_preds = torch.argmax(text_probs, dim=1)
            image_preds = torch.argmax(image_probs, dim=1)

            # Use max class probability as confidence for each head
            text_confidences = torch.max(text_probs, dim=1).values
            image_confidences = torch.max(image_probs, dim=1).values

            # late fusion rule: pick the head with higher confidence for each sample
            combined_preds = []
            for j in range(len(text_preds)):
                if text_confidences[j] > image_confidences[j]:
                    combined_preds.append(text_preds[j].item())
                    all_text_conf.append(text_confidences[j].item())
                    all_image_conf.append(image_confidences[j].item())
                else:
                    combined_preds.append(image_preds[j].item())
                    all_text_conf.append(text_confidences[j].item())
                    all_image_conf.append(image_confidences[j].item())

            # accuracy using combined preds
            combined_preds_tensor = torch.tensor(combined_preds).to(DEVICE)

            total_correct += (combined_preds_tensor == labels).sum().item()
            total_samples += labels.size(0)

            # accumulate for saving
            all_labels.extend(labels.cpu().tolist())
            all_text_preds.extend(text_preds.cpu().tolist())
            all_image_preds.extend(image_preds.cpu().tolist())
            all_combined_preds.extend(combined_preds)

    # compute accuracy
    accuracy = total_correct / total_samples
    print(f"✓ Late Fusion Accuracy: {accuracy*100:.2f}%")

    # Save results to CSV named by the dataset folder (so each Test_X gets its own file)
    test_name = os.path.basename(os.path.normpath(text_dir))
    csv_name = f"late_fusion_results_{test_name}.csv"

    df = pd.DataFrame({
        "label": all_labels,
        "text_pred": all_text_preds,
        "image_pred": all_image_preds,
        "combined_pred": all_combined_preds,
        "text_confidence": all_text_conf,
        "image_confidence": all_image_conf,
    })

    df.to_csv(csv_name, index=False)
    print(f"✓ Saved {csv_name} in current directory")

    return accuracy


In [33]:
# Load the best checkpoint for evaluation
FUSION_CKPT = os.path.join(OUTPUT_DIR, "fusion_ct5p_encoder_epoch_3.pt")

# Reinitialize model
eval_model = FusionClassifier(text_model, vit_model).to(DEVICE)

# Load checkpoint if available
if os.path.exists(FUSION_CKPT):
    state_dict = torch.load(FUSION_CKPT, map_location=DEVICE)
    eval_model.load_state_dict(state_dict, strict=False)
    eval_model.eval()
    print("✓ Evaluation model loaded successfully")
else:
    print(f"Checkpoint not found: {FUSION_CKPT}. Exiting evaluation.")
    raise FileNotFoundError(FUSION_CKPT)

# Test on all test datasets
print("Running evaluation on test datasets...")
accuracies = {}

for i in range(10):
    test_name = f"Test_{i}"
    print(f"\n{test_name}:")
    accuracy = test_on_dataset(
        f"../Text_Files/{test_name}", 
        f"../snapshots/{test_name}", 
        tokenizer, 
        image_processor, 
        eval_model, 
        batch_size=4
    )
    accuracies[test_name] = accuracy

# Print summary
print("\n" + "="*50)
print("SUMMARY OF RESULTS")
print("="*50)
for test_name, acc in accuracies.items():
    print(f"{test_name}: {acc*100:.2f}%")

avg_accuracy = np.mean(list(accuracies.values()))
print(f"\nAverage Accuracy: {avg_accuracy*100:.2f}%")
print("✓ Evaluation completed!")


✓ Evaluation model loaded successfully
Running evaluation on test datasets...

Test_0:
✓ Loaded 1002 samples


Testing: 100%|██████████| 251/251 [00:13<00:00, 18.65it/s]



✓ Late Fusion Accuracy: 21.16%
✓ Saved late_fusion_results_Test_0.csv in current directory

Test_1:
✓ Loaded 1002 samples


Testing: 100%|██████████| 251/251 [00:13<00:00, 18.70it/s]


✓ Late Fusion Accuracy: 25.95%
✓ Saved late_fusion_results_Test_1.csv in current directory

Test_2:
✓ Loaded 1015 samples


Testing:  55%|█████▍    | 139/254 [00:07<00:06, 18.77it/s]



KeyboardInterrupt: 